# Ultron voice - train on Colab
Runtime -> Change runtime type -> **T4 GPU** first. Then run the cells top to bottom.

In [ ]:
import torch; print(torch.cuda.get_device_name(0))
from google.colab import files
up = files.upload()   # choose vtts_colab.zip

In [ ]:
!unzip -q -o vtts_colab.zip
!pip -q install soundfile g2p_en
import nltk
for p in ['averaged_perceptron_tagger','averaged_perceptron_tagger_eng','cmudict']: nltk.download(p, quiet=True)

## Train (acoustic + vocoder in parallel, ~40 min on a T4)
Vocoder: first 6000 steps generator-only (fast), then GAN to 12000.

In [ ]:
import subprocess, time
ac = subprocess.Popen('python -u -m vtts train-acoustic --data data/all --out runs/ultron --steps 8000 --amp > ac.log 2>&1', shell=True)
vo = subprocess.Popen('python -u -m vtts train-vocoder --data data/all --out runs/ultron --steps 12000 --g-only-steps 6000 --small > voc.log 2>&1', shell=True)
while ac.poll() is None or vo.poll() is None:
    time.sleep(90)
    print('ACOUSTIC:', subprocess.getoutput('tail -n 1 ac.log')[:160])
    print('VOCODER :', subprocess.getoutput('tail -n 1 voc.log')[:160])
print('done', ac.returncode, vo.returncode)

If the acoustic log shows `nan`, re-run that job without `--amp`. Otherwise continue.

In [ ]:
from vtts.synth import Synthesizer
from vtts.audio import save_wav
from IPython.display import Audio, display
s = Synthesizer('runs/ultron/acoustic.pt', 'runs/ultron/vocoder.pt')
text = "I am Ultron. There are no strings on me."
for spk, name in enumerate(s.speakers):
    y = s.tts(text, speaker=spk); save_wav(f'{name}.wav', y, s.audio.sr)
    print(name); display(Audio(y, rate=s.audio.sr))

In [ ]:
!zip -j ultron_models.zip runs/ultron/acoustic.pt runs/ultron/vocoder.pt
from google.colab import files; files.download('ultron_models.zip')